In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from src.evaluacion.metricas import evaluar
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from src.evaluacion.metricas import evaluar
from src.features.feature_sets import get_feature_sets, TARGET
from xgboost import XGBRegressor
from src.evaluacion.backtesting import walk_forward
import mlflow

In [ ]:
df =pd.read_parquet("../data/processed/tabla_features.parquet").sort_values("datetime_utc")
df = df[df["entrenable"]]
df_bench = pd.read_parquet("../reports/baseline_walkforward.parquet")



In [ ]:
ultima_fecha = df["datetime_utc"].max()
corte = ultima_fecha - pd.DateOffset(months=6)
train = df[df["datetime_utc"] < corte] 
test = df[df["datetime_utc"] >= corte]
print(len(train))
print(len(test))

In [ ]:
feats = get_feature_sets(df)["predictivo"]

X_train, y_train = train[feats], train[TARGET]
X_test,  y_test  = test[feats],  test[TARGET]

# Controles de seguridad (anti-leakage y anti-NaN)
assert TARGET not in feats and "precio_portugal" not in feats
assert not any(c.endswith("_real") for c in feats)
assert X_train.isna().sum().sum() == 0, "hay NaN en X_train"
print(f"{len(feats)} features | X_train {X_train.shape} | X_test {X_test.shape}")

In [ ]:
modelo = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

resultado_xgb = evaluar(y_test, y_pred)   # orden (y_real, y_pred)
resultado_xgb

In [ ]:
df["meses"] = df["datetime_utc"].dt.to_period("M")
df_wf_xgb = walk_forward(
    df, TARGET,
    lambda tr, te: modelo.fit(tr[feats], tr[TARGET]).predict(te[feats]),
)
print(f"XGB walk-forward → MAE {df_wf_xgb['MAE'].mean():.2f} ± {df_wf_xgb['MAE'].std():.2f}")

In [ ]:
df_wf_xgb["mes"] = df_wf_xgb["mes"].dt.to_timestamp()
# df_wf_xgb = df_wf_xgb.drop("MAPE")

df_wf_xgb = df_wf_xgb.rename(columns={
    "MAE":   "MAE_xgb",
    "RMSE":  "RMSE_xgb",
    "sMAPE": "sMAPE_xgb",
    "MAPE": "MAPE_xgb",
})
df_bench = df_bench.rename(columns={
    "MAE":   "MAE_naive",
    "RMSE":  "RMSE_naive",
    "sMAPE": "sMAPE_naive",
    "MAPE": "MAPE_naive",
})


df_wf_comparacion = pd.merge(df_wf_xgb, df_bench , on = "mes" , how = "inner")
df_wf_comparacion


In [ ]:
# quitar las MAPE (inservibles: ~18% de horas con precio ≤ 0)
df_wf_comparacion = df_wf_comparacion.drop(columns=["MAPE_xgb", "MAPE_naive"])

# cuánto gana XGB cada mes (positivo = XGB mejor que el naive)
df_wf_comparacion["diferencia"] = (
    df_wf_comparacion["mae_naive_d1"] - df_wf_comparacion["MAE_xgb"]
)

# ¿gana más en invierno? (oct-mar)
df_wf_comparacion["invierno"] = df_wf_comparacion["mes"].dt.month.isin([10, 11, 12, 1, 2, 3])
print(df_wf_comparacion.groupby("invierno")["diferencia"].mean())

# corte más nítido: por año (efecto régimen / madurez del train)
print(df_wf_comparacion.groupby(df_wf_comparacion["mes"].dt.year)["diferencia"].mean())

df_wf_comparacion.sort_values("diferencia")   # de peor a mejor para XGB

In [ ]:
d = df_wf_comparacion.sort_values("mes")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(d["mes"], d["mae_naive_d1"], marker="o", color="#e6550d", label="naive D-1")
ax.plot(d["mes"], d["MAE_xgb"],      marker="o", color="#2c7fb8", label="XGBoost")

# verde donde XGB gana (su MAE es menor), rojo donde pierde
ax.fill_between(d["mes"], d["MAE_xgb"], d["mae_naive_d1"],
                where=d["MAE_xgb"] <= d["mae_naive_d1"],
                color="green", alpha=0.15, interpolate=True)
ax.fill_between(d["mes"], d["MAE_xgb"], d["mae_naive_d1"],
                where=d["MAE_xgb"] >  d["mae_naive_d1"],
                color="red", alpha=0.15, interpolate=True)

ax.set_title("Walk-forward: MAE mensual — naive D-1 vs XGBoost")
ax.set_xlabel("Mes"); ax.set_ylabel("MAE (€/MWh)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/figures/walkforward_naive_vs_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_wf_xgb_roll = walk_forward(
    df, TARGET,
    lambda tr, te: modelo.fit(tr[feats], tr[TARGET]).predict(te[feats]),
    ventana=12,
)
print(f"XGB rolling-12 → MAE {df_wf_xgb_roll['MAE'].mean():.2f} ± {df_wf_xgb_roll['MAE'].std():.2f}")

In [ ]:
por_año = pd.DataFrame({
    "naive":     df_wf_comparacion.groupby(df_wf_comparacion["mes"].dt.year)["mae_naive_d1"].mean(),
    "expanding": df_wf_comparacion.groupby(df_wf_comparacion["mes"].dt.year)["MAE_xgb"].mean(),
    "rolling":   df_wf_xgb_roll.groupby(df_wf_xgb_roll["mes"].dt.year)["MAE"].mean(),
})
por_año

In [ ]:
df_wf_xgb_roll = walk_forward(
    df, TARGET,
    lambda tr, te: modelo.fit(tr[feats], tr[TARGET]).predict(te[feats]),
    ventana=24,
)
print(f"XGB rolling-24 → MAE {df_wf_xgb_roll['MAE'].mean():.2f} ± {df_wf_xgb_roll['MAE'].std():.2f}")

In [ ]:
por_año = pd.DataFrame({
    "naive":     df_wf_comparacion.groupby(df_wf_comparacion["mes"].dt.year)["mae_naive_d1"].mean(),
    "expanding": df_wf_comparacion.groupby(df_wf_comparacion["mes"].dt.year)["MAE_xgb"].mean(),
    "rolling":   df_wf_xgb_roll.groupby(df_wf_xgb_roll["mes"].dt.year)["MAE"].mean(),
})
por_año

## Fase 5.1 — XGBoost predictivo: resultados y experimento de ventana

### Modelo base (sin tuning)
- Hiperparámetros de arranque (n_estimators=400, lr=0.05, max_depth=6, subsample/colsample=0.8).
- **Holdout:** MAE 13,61 / RMSE 18,16 (vs naive 15,5 / 23,69).
- **Walk-forward (31 folds):** MAE **15,70 ± 5,29** vs naive **18,33 ± 4,69** → ~14% mejor.
  El holdout era optimista; el número honesto es el walk-forward.

### Hallazgo: la media esconde dos regímenes
- XGB gana en **24/30 meses**, y **más en invierno** (oct-2024 +11,9; oct/nov-2025 +11,5/+10,4) — justo donde el naive era peor.
- **Falla en primavera-2024** (mar −20,5; abr −23,4): el precio se desploma tras la crisis del gas y el modelo, anclado al régimen alto, sobrepredice; el naive se adapta al instante.
- Corte por año (más nítido que el estacional): **2024 = −0,44 · 2025 = +5,48 · 2026 = +2,70**. La debilidad es **temporal (cambio de régimen), no estacional**.

### Experimento expanding vs rolling — hipótesis RECHAZADA
Hipótesis: el train arrastra la crisis del gas 2022-2023 → una ventana **rolling** que olvide lo viejo arreglaría 2024.

| Año | naive | expanding | rolling-12 | rolling-24 |
|---|---|---|---|---|
| 2024 | 18,04 | 18,48 | 19,61 | 18,48 |
| 2025 | 19,91 | 14,43 | 15,67 | 14,12 |
| 2026 | 15,39 | 12,69 | 15,41 | 12,80 |

- rolling-12: peor en los 3 años (y peor que el naive en 2024).
- rolling-24 ≈ expanding (empate dentro del ruido). Caso decisivo = **2026**, único año donde de verdad difieren (rolling-24 ya suelta la crisis): si el histórico viejo estorbara, ganaría ahí — y no lo hace.
- **Conclusión: el histórico viejo NO degrada la predicción. Decisión de producción → ventana EXPANDING** (navaja de Occam: rolling-24 solo iguala añadiendo un hiperparámetro sin beneficio).

In [ ]:
feats_exp = get_feature_sets(df)["explicativo"]

X_train_exp, y_train_exp = train[feats_exp], train[TARGET]
X_test_exp,  y_test_exp  = test[feats_exp],  test[TARGET]


assert TARGET not in feats_exp and "precio_portugal" not in feats_exp   
print("NaN en X_train_exp:", int(X_train_exp.isna().sum().sum()))
print(f"{len(feats_exp)} features | X_train {X_train_exp.shape} | X_test {X_test_exp.shape}")

In [ ]:
modelo_exp = XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", random_state=42, n_jobs=-1,
)
modelo_exp.fit(X_train_exp, y_train_exp)
y_pred_exp = modelo_exp.predict(X_test_exp)

resultado_exp = evaluar(y_test_exp, y_pred_exp)   # orden (y_real, y_pred)
resultado_exp

In [ ]:
df_wf_exp = walk_forward(
    df, TARGET,
    lambda tr, te: modelo_exp.fit(tr[feats_exp], tr[TARGET]).predict(te[feats_exp]),
)
print(f"XGB explicativo walk-forward → MAE {df_wf_exp['MAE'].mean():.2f} ± {df_wf_exp['MAE'].std():.2f}")

In [ ]:
comparacion = pd.DataFrame({
    "MAE_holdout": {
        "naive_d1":    15.5,                         # de Fase 4 (baseline)
        "predictivo":  resultado_xgb["MAE"],
        "explicativo": resultado_exp["MAE"],
    },
    "MAE_walkforward": {
        "naive_d1":    df_bench["mae_naive_d1"].mean(),
        "predictivo":  df_wf_comparacion["MAE_xgb"].mean(),
        "explicativo": df_wf_exp["MAE"].mean(),
    },
})
comparacion.round(2)

## Fase 5.1 — Conclusiones

**Modelo elegido: XGBoost predictivo, ventana expanding.**

| Modelo | MAE holdout | MAE walk-forward |
|---|---|---|
| naive D-1 | 15,5 | 18,33 |
| **XGB predictivo** | **13,61** | **15,70** |
| XGB explicativo | 14,45 | 16,73 |

**Dos hipótesis intuitivas, ambas rechazadas con evidencia:**

1. **Rolling batiría a expanding** (por olvidar la crisis del gas). → FALSO: rolling-12 peor en los 3 años; rolling-24 ≈ expanding. El histórico viejo no estorba → **expanding**.
2. **El explicativo (datos reales) sería un techo superior.** → FALSO: rinde peor que el predictivo. El precio diario se forma **ex-ante** en subasta con las **previsiones**; los valores reales son posteriores y no participaron en formar el precio → ruido. La palanca de negocio es **tener buenas previsiones a tiempo de subasta**, no conocer la realidad.

**Pendiente Fase 5:** intervalos de predicción / cuantiles (5.3), SHAP (interpretabilidad), tuning (Optuna + TimeSeriesSplit, margen esperado pequeño).

In [ ]:
import math
from pathlib import Path

modelo = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

# Tracking directo al fichero SQLite de la RAIZ del proyecto (sin servidor HTTP).
# Ruta ABSOLUTA -> siempre la misma BD, se ejecute desde donde se ejecute.
DB_PATH = (Path("..").resolve() / "mlflow.db").as_posix()
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
mlflow.set_experiment("spotprice-xgboost")

with mlflow.start_run(run_name="xgboost_predictivo_holdout"):
    mlflow.log_params(modelo.get_params())
    mlflow.set_tag("validacion", "holdout_6m")

    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    resultado_xgb = evaluar(y_test, y_pred)

    # Descartamos metricas no finitas: el MAPE es inf (~18% de horas con precio <= 0).
    metricas = {k: v for k, v in resultado_xgb.items() if math.isfinite(v)}
    mlflow.log_metrics(metricas)

    # pip_requirements EXPLICITO -> desactiva la inferencia de entorno de MLflow,
    # que lanza un subproceso (pip) bloqueado por App Control de Windows = cuelgue.
    mlflow.xgboost.log_model(
        modelo,
        name="model",
        pip_requirements=["xgboost", "scikit-learn"],
    )

print("Run registrado. MAE:", round(resultado_xgb["MAE"], 3))


In [ ]:
with mlflow.start_run(run_name="xgboost_predictivo_walkforward"):
    mlflow.log_params(modelo.get_params())
    mlflow.set_tag("validacion", "walk_forward_expanding")

    # 1) el backtesting como siempre — tu funcion NO sabe de MLflow
    df_wf_xgb = walk_forward(
        df, TARGET,
        lambda tr, te: modelo.fit(tr[feats], tr[TARGET]).predict(te[feats]),
    )

    # 2) MAE por fold como SERIE temporal (step = indice del mes)
    for i, fila in df_wf_xgb.reset_index(drop=True).iterrows():
        mlflow.log_metric("MAE_fold", fila["MAE"], step=i)

    # 3) metricas RESUMEN (las comparables entre configuraciones)
    mlflow.log_metric("MAE_wf_mean", df_wf_xgb["MAE"].mean())
    mlflow.log_metric("MAE_wf_std",  df_wf_xgb["MAE"].std())
    mlflow.log_metric("RMSE_wf_mean", df_wf_xgb["RMSE"].mean())

In [ ]:
from mlflow import MlflowClient

with mlflow.start_run(run_name="xgboost_produccion_full"):
    mlflow.log_params(modelo.get_params())
    mlflow.set_tag("validacion", "produccion_full_data")
    # metrica HONESTA de referencia: viene del walk-forward, no de este fit
    mlflow.log_metric("MAE_wf_ref", df_wf_xgb["MAE"].mean())

    # entrenar con TODOS los datos disponibles
    modelo.fit(df[feats], df[TARGET])

    # log + registro en una sola llamada (registered_model_name)
    info = mlflow.xgboost.log_model(
        modelo,
        name="model",
        pip_requirements=["xgboost", "scikit-learn"],
        registered_model_name="spotprice-xgboost",
    )

# marcar esta version como la elegida
client = MlflowClient()
client.set_registered_model_alias("spotprice-xgboost", "champion", info.registered_model_version)
print("Registrada version", info.registered_model_version, "como @champion")